# Lesson 00: The maths behind ML, without the fear

Most of the maths in machine learning is a handful of ideas used over and over: averages, slopes, the chain rule, and "multiply and add". This lesson builds each one from scratch, starting from things you already understand (money, mostly), with small numbers you can check by hand.

**You need:** GCSE maths and basic Python. Nothing else.

**How each idea is taught**
1. **The idea** in plain English, usually with a money example.
2. **Small numbers** you can work out by hand.
3. **The notation**, read out in words.
4. **Code.** Often the clearest way to read a formula is as a loop.
5. **Why ML cares.**

**Symbols used in every lesson**
- ▶ **Run this**: a demo. Just run it and look.
- ✍️ **Your turn**: you write the code.
- 🔮 **Predict**: write your guess in a comment *before* running. Being wrong is the useful part.
- ✅ **Check yourself**: quick questions, with answers folded away underneath.

**Rule for this lesson:** write the functions in `toolkit.py` with plain loops and arithmetic, no NumPy. The point is to see that nothing magic is going on. Afterwards you compare with NumPy, which does the same thing much faster.

Check your work with `uv run pytest lessons/00-maths-toolkit`.

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
import toolkit  # your code; autoreload picks up edits without restarting the kernel

## 1. Reading maths like code

Maths notation is a very compressed programming language. Once you can translate it, most ML formulas turn out to be a loop and some arithmetic.

| Notation | Say it as | In Python |
|---|---|---|
| $x_i$ | "x sub i": item number $i$ of $x$ | `x[i]` |
| $\sum_{i=1}^{n} x_i$ | "the sum of $x_i$ for $i$ from 1 to $n$" | `total = 0`<br>`for xi in x: total += xi` |
| $\frac{1}{n}\sum_{i=1}^{n} x_i$ | "add them all up, divide by how many": the average | `sum(x) / len(x)` |
| $\bar{x}$ | "x bar": the mean of $x$ | `mean(x)` |
| $\hat{y}$ | "y hat": a *prediction* of $y$ | `y_pred` |
| $f(x)$ | "f of x": put $x$ into the function $f$ | `f(x)` |
| $x^2$ | "x squared" | `x ** 2` |
| $\lvert x \rvert$ | "the absolute value of x": drop the minus sign | `abs(x)` |
| $\approx$ | "is approximately" | `math.isclose(a, b)` |
| $w \leftarrow w - 1$ | "w becomes w minus 1" | `w = w - 1` |
| $\frac{dy}{dx}$ | "dy by dx": the slope of $y$ as $x$ changes | section 4 |
| $\frac{\partial L}{\partial w}$ | "partial L by partial w": the slope of $L$ when only $w$ changes | section 5 |
| $\mathbf{a} \cdot \mathbf{b}$ | "a dot b": multiply matching pairs, add up | section 7 |

> **Watch out:** maths usually counts from 1, Python from 0. $x_1$ is `x[0]`. This trips everyone up at least once.

**Greek letters you'll meet:** $\alpha$ alpha and $\eta$ eta (learning rates), $\beta$ beta (a stock's sensitivity to the market), $\theta$ theta ("all the model's parameters"), $\mu$ mu (a mean), $\sigma$ sigma (a standard deviation), $\Sigma$ capital sigma (a sum), $\nabla$ nabla (the gradient).

✍️ **Your turn:** translate this into plain Python (a loop, no NumPy) and work it out for `x = [1, 2, 3, 4, 5]`:

$$\frac{1}{n}\sum_{i=1}^{n} (x_i - 3)^2$$

🔮 Predict first: is the answer bigger or smaller than 3?

In [ ]:
# 🔮 Prediction:

x = [1, 2, 3, 4, 5]


### ✅ Check yourself

1. What is $\sum_{i=1}^{3} 2i$?
2. Say $\hat{y}_i - y_i$ in plain English.

<details>
<summary>Answers (try first)</summary>

1. 2 + 4 + 6 = 12.
2. The prediction for example $i$ minus the true value for example $i$: how far off we were on that example.

</details>

## 2. Averages and spread: mean, variance, standard deviation

**The idea.** Two stocks both average +0.05% a day. One moves about ±0.1% on a typical day; the other moves about ±3%. Same average, very different ride, and the second one is far riskier. One number can't capture both, so we use two:
- the **mean** says what's typical
- the **standard deviation** says how far from typical things usually land

In finance, the standard deviation of a stock's returns is called its **volatility**. It's the most basic measure of risk there is.

In [ ]:
# ▶ Run this: same average, different spread
rng = np.random.default_rng(0)
calm = rng.normal(loc=0.05, scale=0.1, size=250)
wild = rng.normal(loc=0.05, scale=3.0, size=250)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, series, name in zip(axes, [calm, wild], ["Calm stock", "Wild stock"]):
    ax.plot(series, linewidth=0.8)
    ax.axhline(series.mean(), color="black", linestyle="--", label=f"mean = {series.mean():.2f}%")
    ax.set(title=f"{name}: standard deviation = {series.std():.2f}%", xlabel="Day")
    ax.legend()
axes[0].set_ylabel("Daily return (%)")
plt.show()

**By hand.** Take `[2, 4, 4, 4, 5, 5, 7, 9]`.

1. **Mean:** add up to 40, divide by 8 → **5**.
2. **Distance of each value from the mean:** −3, −1, −1, −1, 0, 0, 2, 4.
3. **Square them** (otherwise the negatives and positives cancel out, and they'd always add to 0): 9, 1, 1, 1, 0, 0, 4, 16.
4. **Average of those squares:** 32 / 8 → **variance = 4**.
5. **Square root**, to get back to the original units (otherwise it'd be in "% squared"): **standard deviation = 2**.

**The notation**, now that you know what it does:

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i \qquad\qquad \sigma^2 = \frac{1}{n}\sum_{i=1}^{n} (x_i - \bar{x})^2 \qquad\qquad \sigma = \sqrt{\sigma^2}$$

Read the middle one aloud: "the variance is the average of the squared distances from the mean."

> **n or n − 1?** You'll also see variance divided by $n - 1$, called the *sample* variance. It corrects a slight underestimate when you only have a sample of the data. NumPy's `np.var` divides by $n$ by default; pandas' `.var()` divides by $n - 1$. With lots of data it barely matters, but it's a classic cause of "why don't my numbers match?"

✍️ **Your turn**
1. Implement `mean`, `variance` and `std` in `toolkit.py`. Run `-k "Mean or Variance"`.
2. Check your functions against `np.mean`, `np.var` and `np.std` on the `wild` returns above.

In [ ]:
# Your turn: compare with NumPy


### ✅ Check yourself

1. If you add 10 to every number, what happens to the mean? To the standard deviation?
2. If you double every number?

<details>
<summary>Answers (try first)</summary>

1. The mean goes up by 10. The standard deviation doesn't change: everything moved together, so the spread is the same.
2. Both double.

</details>

## 3. Functions: machines that turn inputs into outputs

**The idea.** A function takes an input and gives back an output. $f(x) = x^2$ takes 3 and gives 9.

In ML, the most important function is the **loss**: you put in the model's settings, and it gives back one number saying how wrong the model is. **Training a model means searching for the input that makes the loss as small as possible.** Most of what follows is about doing that search cleverly.

In [ ]:
# ▶ Run this
x = np.linspace(-2, 6, 200)
plt.figure(figsize=(7, 4))
plt.plot(x, 2 * x + 1, label="2x + 1")
plt.plot(x, x**2, label="x²")
plt.plot(x, (x - 3) ** 2 + 1, label="(x − 3)² + 1")
plt.ylim(-2, 15)
plt.axhline(0, color="grey", linewidth=0.5)
plt.legend()
plt.show()

✍️ 🔮 **Your turn:** where is the lowest point of $(x - 3)^2 + 1$, and what is its value there? Work it out *without* the graph first, then check on the graph. Why is it there?

In [ ]:
# 🔮 Prediction and reasoning:


### ✅ Check yourself

1. Why can't $(x - 3)^2 + 1$ ever be less than 1?

<details>
<summary>Answers (try first)</summary>

1. A square is never negative, so $(x - 3)^2$ is at least 0. It's exactly 0 when $x = 3$, which makes the lowest value $0 + 1 = 1$.

</details>

## 4. Slopes and derivatives: "if I nudge the input, how much does the output move?"

This is the single most important idea in machine learning.

**Straight lines.** The slope is "rise over run". For $y = 2x + 1$, increasing $x$ by 1 increases $y$ by 2, everywhere on the line. The slope is 2.

**Curves.** The slope changes as you move along. $x^2$ is flat near 0 and gets steeper the further out you go. So we ask for the slope *at a particular point*. The trick: zoom in. Any smooth curve looks like a straight line if you zoom in far enough, and that line's slope is the curve's slope at that point.

In [ ]:
# ▶ Run this: zooming in on x² around x = 1
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, width in zip(axes, [2, 0.2, 0.02]):
    x = np.linspace(1 - width, 1 + width, 100)
    ax.plot(x, x**2)
    ax.plot(1, 1, "o", color="black")
    ax.set(title=f"zoom: x from {1 - width:g} to {1 + width:g}")
plt.show()

By the third panel it's a straight line, and its slope is the slope of $x^2$ at $x = 1$.

So to *measure* the slope at $x$: nudge $x$ by a tiny amount $h$, see how much $f$ changes, and divide. Nudging both ways and dividing by the total width $2h$ turns out to be more accurate than nudging one way:

$$\text{slope at } x \approx \frac{f(x + h) - f(x - h)}{2h}$$

That slope is called the **derivative**, written $f'(x)$ or $\frac{df}{dx}$. In words: *how much the output moves per unit of input, right here.*

✍️ **Your turn**
1. Implement `numerical_derivative(f, x, h)`. Run `-k Derivative`.
2. 🔮 Predict the *sign* (positive, negative or zero) of the slope of $x^2$ at $x = -2, 0, 1, 3$. Then measure them. What pattern do the numbers follow?

In [ ]:
# 🔮 Predictions:

# Your turn: slopes of x² at -2, 0, 1, 3


**Shortcut rules.** Mathematicians worked out formulas that give the exact slope without nudging. You don't need many; these cover most of ML:

| Function | Its slope (derivative) | In words |
|---|---|---|
| $c$ (a constant) | $0$ | flat, never changes |
| $a \cdot x$ | $a$ | a straight line's slope is its steepness |
| $x^2$ | $2x$ | steeper the further from 0 (your pattern above) |
| $x^n$ | $n \cdot x^{n-1}$ | the general version of the line above |
| $e^x$ | $e^x$ | its own slope (section 9) |
| $\ln x$ | $1/x$ | |
| $f + g$ | $f' + g'$ | the slope of a sum is the sum of the slopes |

Think of them as **shortcuts that give the same number as the nudge test**. When you're unsure, check one with `numerical_derivative`. ML engineers do exactly this; lesson 01 calls it *gradient checking*.

**Why ML cares: the sign of the slope tells you which way is downhill.**
- Slope positive → moving right goes up → to go down, move left.
- Slope negative → moving right goes down → move right.
- Slope zero → you're at the bottom (or a top, or a flat bit).

✍️ **Your turn:** find the lowest point of $f(x) = (x - 3)^2 + 1$ using *only* slopes, no graph and no algebra:

```
start at x = 0
repeat 20 times:
    x = x - 0.1 * (slope of f at x)
    print x
```

🔮 Predict where `x` ends up. Then try a step size of `1.1` instead of `0.1`, and predict that first too.

You've just written **gradient descent**, the algorithm behind nearly all of modern ML. Lesson 01 does the same thing with two knobs instead of one.

In [ ]:
# 🔮 Prediction:

# Your turn: gradient descent on one knob


### ✅ Check yourself

1. The slope of $f$ at $x$ is −4. To make $f$ smaller, should $x$ go up or down?
2. Why is the step `x - 0.1 * slope` and not `x + 0.1 * slope`?
3. What happens to the size of each step as `x` gets close to the bottom, and why?

<details>
<summary>Answers (try first)</summary>

1. Up. A negative slope means going right goes downhill.
2. We want to move *against* the slope, which is downhill. Adding the slope would walk uphill.
3. The steps shrink, because the slope gets smaller near the bottom and the step is proportional to it. Gradient descent slows down by itself as it arrives.

</details>

## 5. Partial derivatives: more than one knob

**The idea.** Functions in ML have lots of inputs; a model can have millions of knobs. A **partial derivative** asks the same nudge question about one knob while every other knob stays where it is. Think of a mixing desk: nudge one slider, leave the rest alone, and listen to how the sound changes.

**Small example.** $f(a, b) = a^2 + 3b$.
- Nudge $a$ and hold $b$ still: $3b$ doesn't change, so the slope is just that of $a^2$, which is $2a$.
- Nudge $b$ and hold $a$ still: $a^2$ doesn't change, so the slope is 3.

Written: $\frac{\partial f}{\partial a} = 2a$ and $\frac{\partial f}{\partial b} = 3$. The curly $\partial$ just means "only this one moves".

The **gradient** is the list of all the partial derivatives: $\nabla f = \left(\frac{\partial f}{\partial a}, \frac{\partial f}{\partial b}\right)$. It points in the direction that increases $f$ fastest, uphill. So **minus the gradient points downhill**, and that's the direction gradient descent steps in.

In [ ]:
# ▶ Run this: a two-knob bowl, with arrows pointing downhill (minus the gradient)
a, b = np.meshgrid(np.linspace(-2, 4, 200), np.linspace(-5, 1, 200))
f = (a - 1) ** 2 + 2 * (b + 2) ** 2

a_arrows, b_arrows = np.meshgrid(np.linspace(-2, 4, 9), np.linspace(-5, 1, 9))
downhill_a = -2 * (a_arrows - 1)
downhill_b = -4 * (b_arrows + 2)

plt.figure(figsize=(6, 5))
plt.contour(a, b, f, levels=15, cmap="viridis")
plt.quiver(a_arrows, b_arrows, downhill_a, downhill_b, color="tab:red", alpha=0.7)
plt.plot(1, -2, "k*", markersize=12)
plt.xlabel("knob a")
plt.ylabel("knob b")
plt.title("f(a, b) = (a − 1)² + 2(b + 2)²: rings are equal height, star is the bottom")
plt.show()

Every arrow points downhill, and they all lead to the star. Notice the arrows cross the rings at right angles: the steepest way down is always straight across the contour lines.

✍️ **Your turn**
1. Implement `numerical_gradient(f, params, h)`: nudge each parameter in turn, holding the rest still, and return the list of slopes. Run `-k Gradient`.
2. Use it on $f(a, b) = a^2 + 3b$ at $(1, 2)$. Do you get $(2, 3)$?

In [ ]:
# Your turn


<details>
<summary>Hint, if a test fails in a way you don't understand</summary>

Read the failing test's name carefully. If you nudge `params` in place (`params[i] += h`), what state is the caller's list in afterwards?

</details>

### ✅ Check yourself

1. For $f(a, b) = 5a + b^2$, what is $\frac{\partial f}{\partial a}$ at $(2, 3)$? And $\frac{\partial f}{\partial b}$?
2. From $(2, 3)$, which way should each knob move to make $f$ smaller fastest?

<details>
<summary>Answers (try first)</summary>

1. 5 (the slope of $5a$ is 5, and $b^2$ doesn't change when only $a$ moves). And $2b = 6$.
2. Against the gradient $(5, 6)$: decrease both, $b$ a little more than $a$.

</details>

## 6. The chain rule: rates multiply

**The idea, with money.** £1 buys \$1.25, and \$1 buys €0.90. How many euros does £1 buy? $1.25 \times 0.90 = €1.125$. You multiplied the exchange rates along the chain.

The **chain rule** says the same thing about slopes: when one quantity feeds into another, **their slopes multiply**. If $u$ depends on $x$, and $y$ depends on $u$:

$$\frac{dy}{dx} = \frac{dy}{du} \times \frac{du}{dx}$$

In words: *(how fast $y$ moves per $x$) = (how fast $y$ moves per $u$) × (how fast $u$ moves per $x$)*, exactly like (euros per pound) = (euros per dollar) × (dollars per pound).

**Worked example.** $y = (3x - 1)^2$. Break it into a chain:
- inner: $u = 3x - 1$, whose slope is $\frac{du}{dx} = 3$
- outer: $y = u^2$, whose slope is $\frac{dy}{du} = 2u$
- multiply: $\frac{dy}{dx} = 2u \times 3 = 6(3x - 1)$

At $x = 1$: $u = 2$, so the slope is $6 \times 2 = 12$.

✍️ **Your turn**
1. Check the 12 with your `numerical_derivative`.
2. By hand: what's the slope of $y = (2x + 5)^2$ at $x = -1$? Then check it numerically.

**Why ML cares.** A neural network is one long chain: input → layer 1 → layer 2 → … → prediction → loss. To learn, it needs the slope of the loss with respect to every knob in every layer, and the chain rule says to multiply the slopes back along the chain. That's all **backpropagation** is (lesson 08). Lesson 01's loss, $(wx + b - y)^2$, is the worked example above with different letters.

In [ ]:
# Your turn: check the chain rule numerically


### ✅ Check yourself

1. The loss rises 2 units for every unit the prediction rises, and the prediction rises 0.5 units for every unit $w$ rises. How fast does the loss rise per unit of $w$?

<details>
<summary>Answers (try first)</summary>

1. $2 \times 0.5 = 1$ unit per unit of $w$.

</details>

## 7. Vectors and the dot product: multiply pairs, add up

**The idea, with a portfolio.** You hold 10 shares of A, 5 of B and 2 of C. Their prices are £3, £10 and £50. The portfolio is worth

$$10 \times 3 + 5 \times 10 + 2 \times 50 = 180$$

You multiplied matching pairs and added up the results. That's the **dot product**.

A **vector** is just a list of numbers in a fixed order: holdings $= (10, 5, 2)$, prices $= (3, 10, 50)$. Then

$$\text{holdings} \cdot \text{prices} = \sum_{i} \text{holdings}_i \times \text{prices}_i = 180$$

**Why ML cares.** A linear model's prediction is exactly this: *features · weights + bias*. Each feature gets a weight saying how much it matters, and you add it all up. It's the portfolio again, with "how much each feature counts" in place of "how many shares".

✍️ **Your turn:** implement `dot(a, b)` with a loop, raising `ValueError` if the lengths differ. (A dot product of mismatched lists means nothing. Which share has no price?) Run `-k Dot`, then compare with NumPy's `a @ b`.

In [ ]:
# Your turn


### ✅ Check yourself

1. $(1, 2, 3) \cdot (4, 5, 6)$?
2. What is $\mathbf{x} \cdot (1, 1, 1)$ for any three numbers $\mathbf{x}$? And $\mathbf{x} \cdot (\frac13, \frac13, \frac13)$?

<details>
<summary>Answers (try first)</summary>

1. 4 + 10 + 18 = 32.
2. The sum of $\mathbf{x}$. And the mean of $\mathbf{x}$: an average is just a dot product with equal weights.

</details>

## 8. Matrices and shapes: a whole spreadsheet at once

**The idea.** A **matrix** is a table of numbers, like a spreadsheet where each **row is one example** (a day, a customer, a transaction) and each **column is one feature**. Its **shape** is (rows, columns): 250 trading days with 2 features each has shape (250, 2).

**Matrix × vector = one dot product per row.** If $X$ holds every example's features and $\mathbf{w}$ holds the weights, then $X\mathbf{w}$ is a prediction for every example in one go:

$$\begin{pmatrix} 1 & 2 \\ 3 & 4 \\ 5 & 6 \end{pmatrix} \begin{pmatrix} 10 \\ 1 \end{pmatrix} = \begin{pmatrix} 1 \times 10 + 2 \times 1 \\ 3 \times 10 + 4 \times 1 \\ 5 \times 10 + 6 \times 1 \end{pmatrix} = \begin{pmatrix} 12 \\ 34 \\ 56 \end{pmatrix}$$

**The shape rule:** (n, d) @ (d,) → (n,). The sizes next to each other must match (each row has $d$ numbers, so it needs $d$ weights), and you get one answer per row.

**Transpose**, written $X^\top$ (`X.T`), flips rows and columns: (250, 2) becomes (2, 250). You'll see it in lesson 01's gradient formula. It's how you go back the other way, from "one number per example" to "one number per feature".

✍️ **Your turn:** implement `matvec(A, x)` using your `dot`, raising `ValueError` when the shapes don't fit. Run `-k Matvec`.

In [ ]:
# Your turn


✍️ 🔮 **Shape quiz.** Predict each shape (or an error), *then* check it with NumPy using `np.ones(...)` arrays:
1. (100, 3) @ (3,)
2. (3, 100) @ (100,)
3. (100, 3) @ (100,)
4. `X.T` where `X` has shape (100, 3)

For any that error, read the whole error message. Learning to read shape errors will save you hours: they're the most common bug in ML code.

In [ ]:
# 🔮 Predictions:
# 1.
# 2.
# 3.
# 4.


### ✅ Check yourself

1. `X` has shape (500, 4). How many examples are there? How many features? What shape must `w` be, and what shape is `X @ w`?

<details>
<summary>Answers (try first)</summary>

1. 500 examples, 4 features. `w` must have shape (4,), and `X @ w` has shape (500,): one prediction per example.

</details>

## 9. Exponentials and logs: growth, and its undo button

You'll need these from lesson 02 onwards.

**Exponentials, with money.** £100 at 10% a year becomes $100 \times 1.1^n$ after $n$ years. Growth that multiplies by the same factor every step is **exponential**.

The special number $e \approx 2.718$ is what £1 grows to in a year at 100% interest compounded continuously: every second, every instant. See it appear:

In [ ]:
# ▶ Run this: £1 at 100% interest, compounded more and more often
for n in [1, 2, 12, 365, 10_000, 1_000_000]:
    print(f"compounded {n:>9,} times a year: £{(1 + 1 / n) ** n:.6f}")
print(f"{'e':>34} = {math.e:.6f}")

**Logs undo exponentials.** $\ln(e^x) = x$. The **natural log** $\ln$ answers "how many times do I have to multiply by $e$ to get this number?"

Its superpower: **it turns multiplying into adding.** $\ln(a \times b) = \ln a + \ln b$.

That's why finance uses **log returns**. Returns compound by multiplying: a +10% day then a −10% day leaves you at $1.1 \times 0.9 = 0.99$, down 1%, not back to even. Log returns just add up, which makes them far easier to work with.

✍️ 🔮 **Your turn:** a stock goes up 50%, then down 50%. Predict where it ends up. Then compute it two ways: by multiplying the simple returns, and by adding the log returns (`np.log(1.5) + np.log(0.5)`, then `np.exp` of the total to turn it back).

In [ ]:
# 🔮 Prediction:

# Your turn


**Why ML cares.** Lesson 02 uses $e$ to squash any number into a probability between 0 and 1, and logs to measure how wrong a probability was. Logs also stop tiny probabilities from multiplying down to exactly 0 inside a computer, which happens far sooner than you'd think.

### ✅ Check yourself

1. What is $\ln(1)$?
2. What is $\ln(e^3)$?

<details>
<summary>Answers (try first)</summary>

1. 0. You multiply by $e$ zero times to get 1.
2. 3.

</details>

## 10. Wrap-up

Fill in **My cheat sheet** in this lesson's README: one line per idea, in your own words, with a tiny example. If you can't write a line, that's the section to reread.

Then add the new terms to `GLOSSARY.md`.

Every later lesson links back here when it uses one of these ideas, so this is the page to come back to whenever a formula stops making sense.